# Kernel Memory & Cache Cleanup Utility

When working with heavy datasets (like WESAD raw `.pkl` files, multi-subject feature extraction, large Pandas DataFrames, Dask arrays, and ML models), RAM usage can spike, leading to VS Code / Jupyter kernel crashes (OOM errors).

This notebook provides tools to:
1. **Monitor RAM / Memory Usage**: Check current system and process memory consumption.
2. **Run Garbage Collection**: Force Python's `gc` module to clean up unreferenced objects and cyclic references.
3. **Purge Large Workspace Variables**: Inspect large variables currently loaded in memory and optionally delete them.
4. **Clear IPython / Jupyter Cache & Outputs**: Reset kernel state or clear execution output buffers.

In [1]:
import gc
import sys
import os

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False

def get_memory_usage():
    if PSUTIL_AVAILABLE:
        process = psutil.Process(os.getpid())
        mem_info = process.memory_info()
        rss_mb = mem_info.rss / (1024 * 1024)
        vms_mb = mem_info.vms / (1024 * 1024)
        
        vm = psutil.virtual_memory()
        print(f"=== System Memory ===")
        print(f"Total RAM: {vm.total / (1024**3):.2f} GB")
        print(f"Available RAM: {vm.available / (1024**3):.2f} GB ({vm.percent}% used)")
        print(f"\n=== Current Python Process (PID {os.getpid()}) ===")
        print(f"RSS (Resident Set Size): {rss_mb:.2f} MB")
        print(f"VMS (Virtual Memory Size): {vms_mb:.2f} MB")
    else:
        print("psutil is not installed. To see detailed system memory, run: pip install psutil")
        print(f"Python process ID: {os.getpid()}")

get_memory_usage()

=== System Memory ===
Total RAM: 15.74 GB
Available RAM: 2.72 GB (82.7% used)

=== Current Python Process (PID 29900) ===
RSS (Resident Set Size): 61.15 MB
VMS (Virtual Memory Size): 73.04 MB


## 1. Force Python Garbage Collection
Python automatically manages memory, but circular references or large cached objects might linger until explicit garbage collection (`gc.collect()`) is triggered.

In [2]:
print("Running garbage collection...")
collected = gc.collect()
print(f"Garbage collector: collected {collected} objects.")

Running garbage collection...
Garbage collector: collected 0 objects.


## 2. Inspect Large Variables in Current Workspace
This cell lists all variables currently loaded in the interactive namespace, sorted by their memory size. You can delete heavy variables (like raw dataframes, pickles, or large models) to free RAM.

In [3]:
import pandas as pd
import numpy as np

def show_large_variables(top_n=15):
    print(f"{'Variable Name':<25} {'Type':<20} {'Size (MB)':<12}")
    print("-" * 60)
    
    var_sizes = []
    for name, val in list(globals().items()):
        if name.startswith('_') or name in ('In', 'Out', 'get_ipython', 'sys', 'os', 'gc', 'psutil', 'pd', 'np', 'plt'):
            continue
        try:
            size_bytes = sys.getsizeof(val)
            if hasattr(val, 'memory_usage'):
                if callable(val.memory_usage):
                    size_bytes = val.memory_usage(deep=True)
                    if isinstance(size_bytes, pd.Series):
                        size_bytes = size_bytes.sum()
            elif hasattr(val, 'nbytes'):
                size_bytes = val.nbytes
            
            var_sizes.append((name, type(val).__name__, size_bytes))
        except Exception:
            pass
            
    var_sizes.sort(key=lambda x: x[2], reverse=True)
    for name, vtype, size in var_sizes[:top_n]:
        print(f"{name:<25} {vtype:<20} {size / (1024*1024):<12.2f}")

show_large_variables()

Variable Name             Type                 Size (MB)   
------------------------------------------------------------
open                      function             0.00        
get_memory_usage          function             0.00        
show_large_variables      function             0.00        
exit                      ZMQExitAutocall      0.00        
quit                      ZMQExitAutocall      0.00        
PSUTIL_AVAILABLE          bool                 0.00        
collected                 int                  0.00        


## 3. Clear Heavy Variables (Optional)
If you want to clear specific large variables (like `raw_data`, `forked_features_df`, etc.), you can delete them and run garbage collection again.

Uncomment and modify the lines below to delete heavy objects from memory:

In [4]:
# Example: delete heavy variables if they exist in your workspace
heavy_vars = ['raw_data', 'forked_features_df', 'df', 'baseline', 'stress', 'amusement']

deleted_count = 0
for var in heavy_vars:
    if var in globals():
        del globals()[var]
        print(f"Deleted variable: {var}")
        deleted_count += 1

if deleted_count > 0:
    gc.collect()
    print(f"\nCleaned up {deleted_count} variables and ran garbage collection.")
else:
    print("No specified heavy variables found in global scope.")

No specified heavy variables found in global scope.


## 4. Clear Jupyter Output Cache & Reset Kernel
In VS Code, cell outputs (especially large plots, dataframes, or print logs) can consume significant RAM inside the Jupyter extension process.
- **To clear cell outputs in VS Code**: Right-click in the notebook and select **"Clear All Outputs"** (or click the Clear Outputs icon in the toolbar).
- **To restart the kernel**: Click the **"Restart Kernel"** button at the top right of the notebook toolbar (or run `%reset -f` below to clear all user namespaces).

In [5]:
# Uncomment below to reset IPython namespace entirely (equivalent to restarting kernel variables)
# %reset -f

print("Memory cleanup complete. Check updated memory usage below:")
get_memory_usage()

Memory cleanup complete. Check updated memory usage below:
=== System Memory ===
Total RAM: 15.74 GB
Available RAM: 2.70 GB (82.9% used)

=== Current Python Process (PID 29900) ===
RSS (Resident Set Size): 92.85 MB
VMS (Virtual Memory Size): 450.24 MB
